# EEG Signal Processing Toolbox - DSP Engine Demonstration

This notebook serves as an interactive demonstration of the digital signal processing (DSP) architecture powering the **EEG Toolbox**. The pipeline demonstrates raw physiological data extraction, artifact removal via infinite impulse response (IIR) notch filtering, and a comparative mathematical analysis of finite impulse response (FIR) filter designs applied to specific neural frequency bands.

### 1. Environment Setup & Data Loading
First, we initialize the environment and load a standard European Data Format (`.edf`) file from the sample repository.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import pyedflib
import data_loader
from data_displayer import display_data, compare_filter
import signal_processing

sample_dir = "EEG_Samples"
files = [f for f in os.listdir(sample_dir) if f.endswith('.edf')]

print(file_names := f"EEG files available: {files}")

if files:
    file_path = os.path.join(sample_dir, files[9])
    f_edf = pyedflib.EdfReader(file_path)
    print(f"EEG signal '{files[9]}' loaded succesfully")

### 2. Signal Extraction and Metadata Parsing
With the `.edf` file loaded, we extract the core data matrix and its associated metadata, including the anatomical channel labels, sampling frequency ($f_s$), and total recording duration.

In [ ]:
data_matrix, n_channels, channel_names, freq, time, dimension_unit = data_loader.load_edf_data(file_path)
print(f"Found {n_channels} available channels: " + f"{channel_names}")
print(f"Sampling frequency: {freq} Hz")
print(f"Sampling time: {time} s")


### 3. Unfiltered Signal Exploration (Time Domain)
To visually evaluate the filtering pipeline, we isolate a single EEG channel. Here, we select the **Afz** (Antero-frontal) channel. The raw signal is plotted below to highlight baseline wander and high-frequency artifacts characteristic of unconditioned neurophysiological recordings.

In [ ]:
channel_selected = 27
name_selected = channel_names[channel_selected-1]
data = data_matrix[channel_selected-1, :]
print(f"Selected channel {channel_selected}: {name_selected}")

fig_unfiltered = display_data(data, name_selected, time, freq, dimension_unit, state="default")

plt.show(fig_unfiltered)

### 4. Powerline Artifact Removal (IIR Notch Filter)
A common source of contamination in EEG recordings is electromagnetic interference from the power grid. We apply an IIR Notch Filter centered at $60\text{ Hz}$ using a high quality factor ($Q = 1000$). This ensures a highly specific attenuation of the powerline artifact while strictly preserving the integrity and power of the adjacent neural frequency bands.

In [ ]:
print("Removing current artifact")

data_clean = signal_processing.Current_Remover(data, freq, f_remove=60.0, Q=1000.0)

fig_clean = display_data(data_clean, name_selected, time, freq, dimension_unit, "default")

plt.show(fig_clean)

### 5. FIR Filter Design and Comparative Analysis
Next, we design a bandpass filter targeting the **Theta ($\theta$) band (3-7 Hz)**, which is often associated with cognitive processing and sleep states. 

To ensure strict linear phase (constant group delay) and minimize signal distortion, we evaluate three distinct FIR filter design algorithms:
*   **Window Method** (using a Hamming/Hanning window)
*   **Equiripple** (Parks-McClellan algorithm for minimax error)
*   **Least Squares** (minimizing the integral of the squared error)

The filters are evaluated with identical transition widths ($0.2\text{ Hz}$) and a high tap order ($N = 1001$) to visually assess the engineering trade-offs between transition steepness, stopband attenuation, and passband ripple.

In [ ]:
trans_width = 0.2
order = int(1e3)
f_cut_l = 3.0
f_cut_h = 7.0
methods = ["Window Method", "Equiripple", "Least Squares"]
filter_selected = "Bandpass Filter"

print("Filtering in \u03B8 band (3-7 Hz) using Hamming Window, Equiripple, Least Sqares")
print("Parameters have been equally set")
print(f"Transition Width: {trans_width} Hz")
print(f"Order: {order+1} Taps")

Fir_window = signal_processing.Fir_designer(freq, order, filter_selected, methods[0], f_cut_l, f_cut_h, trans_width, window_selected="Hanning", beta = None)
Fir_equiripple = signal_processing.Fir_designer(freq, order, filter_selected, methods[1], f_cut_l, f_cut_h, trans_width, window_selected=None, beta=None)
Fir_lsquares = signal_processing.Fir_designer(freq, order, filter_selected, methods[2], f_cut_l, f_cut_h, trans_width, window_selected=None, beta=None)

print("\nComparing All FIR Filters")

fig_Fir_compare = compare_filter(Fir_window, Fir_equiripple, Fir_lsquares, filter_selected, freq)
plt.show(fig_Fir_compare)


### 6. Application of the Optimal Filter
Based on the comparative Bode analysis above, we select the **Least Squares** FIR design for its optimal balance of attenuation and passband flatness. We convolute our notch-filtered EEG data with the selected FIR taps to safely isolate the $\theta$-band oscillations.

In [ ]:
method_selected = methods[2]
data_filtered = signal_processing.Filter_data(data, Fir_lsquares)
fig_data_filtered = display_data(data_filtered, name_selected, time, freq, dimension_unit, state="filtered")
print(f"Method selected: {method_selected}")
plt.show(fig_data_filtered)

### Conclusion
The resulting waveform successfully isolates the target neurophysiological activity, free from powerline noise and out-of-band artifacts, thereby demonstrating the stability and mathematical reliability of the underlying DSP architecture.